In [ ]:
pip install -U weaviate-client

In [ ]:
import weaviate
import json
from weaviate.classes.init import Auth
from weaviate.classes.config import Configure,Property,DataType
from weaviate.classes.query import Filter
from typing import Type,get_type_hints,List,get_origin, get_args
from pydantic import BaseModel,Field
from kaggle_secrets import UserSecretsClient
from sentence_transformers import SentenceTransformer
import numpy as np

In [ ]:
user_secret = UserSecretsClient()
weaviate_key = user_secret.get_secret("weaviate_key") # Weaviate URL: "REST Endpoint" in Weaviate Cloud console
weaviate_url = user_secret.get_secret("weaviate_url") # Weaviate API key: "ADMIN" API key in Weaviate Cloud console

In [ ]:
class FAQ(BaseModel):
    question: str = Field(description="shipping and billing questions")
    answer: str=Field(description="shipping and billing answers")
    tags: List[str]= Field(description="tags for related customer questions such as payment options, refunding process, billing and shipping information")

def convert_schema_to_weaviate(model:BaseModel) ->List[Property]:
    "Convert Pydantic model fields to Weaviate properties with robust type handling."
    
    # Predefined type mapping 
    TYPE_MAPPING = {
        str: DataType.TEXT,
        int: DataType.INT,
        float: DataType.NUMBER,
        bool: DataType.BOOL
    }

    properties = []

    def process_schema_field(name: str, field, prefix: str = "") -> None:
        """Recursively process Pydantic fields with proper type resolution"""
        nonlocal properties
        field_type = field.annotation
        field_desc = getattr(field, 'description', '')
        
        # Handle container types
        origin_type = get_origin(field_type)
        if origin_type is not None:
            # Handle List[str]
            if origin_type is list:
                item_type = get_args(field_type)[0]
                if item_type is str:
                    properties.append(Property(
                        name=f"{prefix}{name}",
                        description=field_desc,
                        data_type=DataType.TEXT_ARRAY
                    ))
                    return
                raise TypeError(f"Unsupported list item type: {item_type} (only List[str] allowed)")
            
            # Handle Optional/Union types
            elif origin_type in (Union, Optional):
                non_none_types = [t for t in get_args(field_type) if t is not type(None)]
                if len(non_none_types) == 1:
                    process_schema_field(name, field, prefix)  # Reprocess with inner type
                    return
                raise TypeError(f"Complex Union types not supported: {field_type}")
            
            # Handle Dict (store as JSON string)
            elif origin_type is dict:
                properties.append(Property(
                    name=f"{prefix}{name}",
                    description=field_desc,
                    data_type=DataType.TEXT
                ))
                return
        
        # Handle nested models
        if hasattr(field_type, "model_fields"):
            for nested_name, nested_field in field_type.model_fields.items():
                process_schema_field(nested_name, nested_field, prefix=f"{prefix}{name}_")
            return
        
        # Handle basic types
        if field_type in TYPE_MAPPING:
            properties.append(Property(
                name=f"{prefix}{name}",
                description=field_desc,
                data_type=TYPE_MAPPING[field_type]
            ))
        else:
            raise TypeError(f"Unsupported field type: {field_type}")

    # Process all fields in the model
    for name, field in model.model_fields.items():
        process_schema_field(name, field)
    
    return properties

In [ ]:
def build_weaviate_collection(
    client:weaviate.WeaviateClient,
    model: BaseModel,
    class_name: str,
    class_description: str="",
    vector_index_config: Configure=Configure.VectorIndex.hnsw()
):
    "This is to create weaviate collection class from pydantic"
    properties = convert_schema_to_weaviate(model)

    # Check if there is an existing collection class
    if client.collections.exists(class_name):
        print(f"Collection exists deleting it...", {class_name})
        client.collections.delete(class_name)
    
    client.collections.create(
        class_name,
        description=class_description,
        vector_index_config=vector_index_config,
        properties=properties
    )

    print(f"Created Collection Class: {class_name}")

In [ ]:
def get_embedding(text: str,embedder: SentenceTransformer) ->list:
    "Generate normalized embedding for text"
    embedding = embedder.encode([text])[0]
    vector = np.array(embedding).astype("float32")
    return (vector / np.linalg.norm(vector)).tolist()

In [ ]:
def load_faqs_to_weaviate(
    collection_name: str,
    embedder: SentenceTransformer,
    file_path:str,
    client: weaviate.Client
)->None:
    "Embed and load FAQs data into existing Weaviate Collection Class using Sentence Transformer"

    try:
        #Load FAQs from JSON file
        with open(file_path, 'r', encoding='utf-8') as f:
            faqs: List[Dict] = json.load(f)
        print(f"Loaded {len(faqs)} FAQs from {file_path}")

        collection = client.collections.get(collection_name)
        successful_count=0
        failed_count=0

        print(f"Generating embeddings and importing FAQs for collection class.....{collection_name}")

        with collection.batch.dynamic() as batch:
            for i, faq in enumerate(faqs, 1):
                try:
                   # Extract fields from your JSON format
                    question = faq.get('question', '').strip()
                    answer = faq.get('answer', '').strip()
                    category = faq.get('category', 'General').strip()
                    subcategory = faq.get('subcategory', '').strip()
                    tags = faq.get('tags', [])

                    # Skip if essential fields are missing
                    if not question or not answer:
                        print(f"Skipping FAQ {i}: Missing question or answer")
                        failed_count += 1
                        continue

                    # Generate embedding using both question and answer for better retrieval
                    text_to_embed = f"Question: {question} Answer: {answer}"
                    embedding = get_embedding(text_to_embed,embedder)
                    #embedding = embedder.encode([text_to_embed])[0].tolist()

                    # Prepare properties object matching JSON structure
                    properties={
                        "question": question,
                        "answer": answer,
                        "category": category,
                        "subcategory": subcategory,
                        "tags": tags,
                        "faq_id": f"faq{i}" 
                    }

                    #Add to Batch
                    batch.add_object(
                        properties = properties,
                        vector = embedding
                    )
                    successful_count +=1
                    
                except Exception as e:
                    print(f"Error processing FAQ : {e}")
                    failed_count+=1   
        print(f"Successfully Loaded {successful_count} FAQs into collection{collection_name}")
           
    except FileNotFoundError:
        print(f"File Not Found:{file_path}")
        raise
        
    except json.JSONDecodeError as e:
        print(f"JSON decode error:{e}")
        raise
        
    except Exception as e:
        print(f"Error loading FAQs to Weaviate:{e}")
        raise

In [ ]:
#Intialize weaviate client
client = weaviate.connect_to_weaviate_cloud(
    cluster_url= weaviate_url,
    auth_credentials= Auth.api_key(weaviate_key)
)

In [ ]:
name = "ecommerce_faqs"
desc = "Shipping,Billing, Customer queries"
print(client.collections.exists(name))

In [ ]:
build_weaviate_collection(client, FAQ, name, desc)

In [ ]:
import os
for dirname,_,filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname,filename))

In [ ]:
# Initialize embedder
embedder = SentenceTransformer('#sentence-transformers/all-MiniLM-L6-v2')

In [ ]:
load_faqs_to_weaviate(
    collection_name="ecommerce_faqs",
    embedder= embedder,
    file_path= "/kaggle/input/shipping-data-set/shipping_billing_faqs.json",
    client=client
)

In [ ]:
query= "Explain me about my delayed order"#"Why is my order delayed?"#"When will my order arrive?"
collection = client.collections.get("ecommerce_faqs")

results= collection.query.hybrid(
    query=query,
    vector=get_embedding(query,embedder),
    limit=3,
    return_properties=["answer","question"],
    return_metadata=["score"],
).objects

print(results)

In [ ]:
for i,result in enumerate(results,1):
    score= getattr(result.metadata,"score","N/A")
    question= result.properties.get("question","N/A")
    answer=result.properties.get("answer", "No answer available")
    print(f"I:{i}, Score{score}, Question={question}, Answer={answer}")
    print("****************")